In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Python Library:


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Data: 

In [ ]:
train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv',header=None)
test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv',header=None)
train_labels = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv',header=None)

# Data Analysis:

In [ ]:
train.info()

In [ ]:
train.shape

In [ ]:
train.isna().sum().sum()

In [ ]:
test.isna().sum().sum()

In [ ]:
train.head()

# Visulization :

In [ ]:
plt.figure(figsize=(10, 6))
for col in train.columns:
    plt.plot(train.index, train[col], label=f'Column {col}')
#plt.legend()
plt.title('Line Plot of Data Columns')
plt.xlabel('Index')
plt.ylabel('Value')
plt.show()

In [ ]:
# Split into two parts
df_part1 = train.iloc[:, :20]
df_part2 = train.iloc[:, 20:]

# Plot part 1
plt.figure(figsize=(16, 10))
corr_part1 = df_part1.corr()
sns.heatmap(corr_part1, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlation Matrix Heatmap - Part 1')
plt.show()

# Plot part 2
plt.figure(figsize=(16, 10))
corr_part2 = df_part2.corr()
sns.heatmap(corr_part2, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlation Matrix Heatmap - Part 2')
plt.show()

In [ ]:
# Replace infinite values with NaN
train.replace([np.inf, -np.inf], np.nan, inplace=True)

# Create the swarm plot
plt.figure(figsize=(10, 6))
sns.stripplot(data=train, jitter=True)
plt.title('Strip Plot of Data Columns')
plt.show()

In [ ]:
x,y = train,train_labels
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

# ML Models:

### KNN model:

In [ ]:
kn_model = KNeighborsClassifier()
kn_model.fit(X_train,y_train.values.ravel())

In [ ]:
pred = kn_model.predict(X_test)
model_accuracy = accuracy_score(y_test,pred)

print(f"Model accuracy: {model_accuracy * 100:.2f}%")

### LogisticRegression model:

In [ ]:
lr_model = LogisticRegression()
lr_model.fit(X_train,y_train.values.ravel())

In [ ]:
lr_pred = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test,lr_pred)

print(f'Lr model Accuracy: {lr_accuracy * 100:.2f}%')

### SVC model with Grid_search:

In [ ]:

y_train_array = y_train.values.ravel() 

param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'gamma': ['scale', 'auto']
}

# SVC model
svc_model = SVC()


grid_search = GridSearchCV(svc_model, param_grid, cv=5)
grid_search.fit(X_train, y_train_array)

best_svc_model = grid_search.best_estimator_

y_pred = best_svc_model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"SVC model accuarcy: {accuracy * 100:.2f}%")
print(f"The best parameters: {grid_search.best_params_}")

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=best_svc_model.classes_,
            yticklabels=best_svc_model.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of SVC model: {accuracy * 100:.2f}%")

### Cat model:

In [ ]:
cat_model = CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, verbose=0,loss_function='Logloss', random_seed=42)
cat_model.fit(X_train,y_train.values.ravel())

In [ ]:
cat_pred = cat_model.predict(X_test)

cat_accuracy = accuracy_score(y_test,cat_pred)
print(f"Cat Model accuracy: {cat_accuracy * 100:.2f}%")

In [ ]:
test_pred = grid_search.predict(test) 
solution = pd.DataFrame({'Id': range(1, len(test_pred) + 1),
               'Solution': test_pred})

In [ ]:
solution.to_csv('solution.csv', index=False)